# LSTM - Sekvencijalni model

Treca komponenta hibridnog modela. LSTM uhvata sekvencijalne zavisnosti koje LightGBM ne vidi direktno.

- Sekvenca: 14 dana -> predikcija sljedeceg dana
- Features: 12 (sales_log, promocija, nafta, kalendar, Prophet komponente)
- Trening: 2016-01-01 do 2017-07-31 (ograniceno za memoriju)
- Output: `models/lstm_val_preds.parquet`, `models/lstm_best.pt`

In [ ]:
import sys, subprocess, os

# Fix: Windows konflikt izmedju PyTorch i numpy/MKL OpenMP runtime-a
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

try:
    import torch
    print(f'PyTorch {torch.__version__} vec instaliran.')
except ImportError:
    print('Instaliram PyTorch (CPU verzija)...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        'torch', '--index-url', 'https://download.pytorch.org/whl/cpu', '-q'
    ], check=True)
    import torch
    print(f'PyTorch {torch.__version__} instaliran.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

PROCESSED  = '../data/processed/'
MODELS     = '../models/'
VAL_START  = pd.Timestamp('2017-08-01')
TRAIN_FROM = pd.Timestamp('2016-01-01')  # samo posljednje 1.5 god. za trening sekvenci
SEQ_LEN    = 14
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Ucitavanje podataka

In [ ]:
train_df = pd.read_parquet(PROCESSED + 'train_features.parquet')
val_df   = pd.read_parquet(PROCESSED + 'val_features.parquet')

# Spajamo da imamo pun kontekst pri gradnji sekvenci za val
all_df = (
    pd.concat([train_df, val_df])
    .sort_values(['store_nbr', 'family', 'date'])
    .reset_index(drop=True)
)

print(f'Train: {train_df.shape}')
print(f'Val:   {val_df.shape}')
print(f'Ukupno: {all_df.shape}')

## 2. Features i normalizacija

In [ ]:
SEQ_FEATURES = [
    'sales_log',           # autoregresivni input: sama serija
    'onpromotion',
    'oil_price',
    'dayofweek',
    'is_weekend',
    'is_payday',
    'is_national_holiday',
    'is_local_holiday',
    'prophet_trend',
    'prophet_weekly',
    'prophet_yearly',
    'prophet_yhat',
]
N_FEATURES = len(SEQ_FEATURES)
print(f'Broj features: {N_FEATURES}')

In [ ]:
# Fit skalera samo na trening podacima
scaler = StandardScaler()
train_mask = all_df['date'] < VAL_START
scaler.fit(all_df.loc[train_mask, SEQ_FEATURES])

all_scaled = all_df.copy()
all_scaled[SEQ_FEATURES] = scaler.transform(all_df[SEQ_FEATURES])

sales_log_idx = SEQ_FEATURES.index('sales_log')
print(f'sales_log: mean={scaler.mean_[sales_log_idx]:.3f}, std={scaler.scale_[sales_log_idx]:.3f}')

## 3. Gradnja sekvenci

Za svaku grupu (store_nbr, family): gradimo sekvence od SEQ_LEN dana.
Za val sekvence koristimo stvarne historijske vrijednosti (teacher forcing evaluacija).
Trening sekvence ogranicavamo od 2016-01-01 kako bismo smanjili potrosnju memorije.

In [ ]:
def build_sequences_for_group(group_df, seq_len):
    arr   = group_df[SEQ_FEATURES].values.astype(np.float32)
    tgt   = group_df['sales_log'].values.astype(np.float32)  # scaled
    dates = group_df['date'].values
    X, y, out_dates = [], [], []
    for i in range(seq_len, len(arr)):
        X.append(arr[i - seq_len:i])
        y.append(tgt[i])
        out_dates.append(dates[i])
    if not X:
        return None, None, None
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), np.array(out_dates)


train_X_list, train_y_list = [], []
val_X_list,   val_y_list   = [], []
val_stores_list, val_families_list, val_dates_list = [], [], []

groups = all_scaled.groupby(['store_nbr', 'family'])
total  = len(groups)

for idx, (key, group) in enumerate(groups):
    store, family = key
    group = group.sort_values('date')

    X, y, dates = build_sequences_for_group(group, SEQ_LEN)
    if X is None:
        continue

    is_val   = dates >= np.datetime64(VAL_START)
    is_train = (dates >= np.datetime64(TRAIN_FROM)) & ~is_val

    if is_train.any():
        train_X_list.append(X[is_train])
        train_y_list.append(y[is_train])

    if is_val.any():
        val_X_list.append(X[is_val])
        val_y_list.append(y[is_val])
        n = is_val.sum()
        val_stores_list.extend([store]  * n)
        val_families_list.extend([family] * n)
        val_dates_list.extend(dates[is_val])

    if (idx + 1) % 400 == 0:
        print(f'  {idx+1}/{total} grupa...')

X_train = np.concatenate(train_X_list, axis=0)
y_train = np.concatenate(train_y_list, axis=0)
X_val   = np.concatenate(val_X_list,   axis=0)
y_val   = np.concatenate(val_y_list,   axis=0)

print(f'\nX_train: {X_train.shape}  ({X_train.nbytes / 1e6:.0f} MB)')
print(f'X_val:   {X_val.shape}')

## 4. Dataset i DataLoader

In [ ]:
class SalesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):         return len(self.y)
    def __getitem__(self, i):  return self.X[i], self.y[i]


BATCH_SIZE   = 2048
train_loader = DataLoader(SalesDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(SalesDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')

## 5. Model

In [ ]:
class SalesLSTM(nn.Module):
    def __init__(self, n_features, hidden=128, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            n_features, hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


model = SalesLSTM(N_FEATURES).to(DEVICE)
print(model)
print(f'\nParametri: {sum(p.numel() for p in model.parameters()):,}')

## 6. Treniranje

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
criterion = nn.MSELoss()

EPOCHS     = 30
PATIENCE   = 6
best_loss  = float('inf')
best_epoch = 0
history    = {'train': [], 'val': []}

for epoch in range(EPOCHS):
    # --- trening ---
    model.train()
    train_losses = []
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_losses.append(loss.item())

    # --- validacija ---
    model.eval()
    val_losses = []
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            val_losses.append(criterion(model(Xb), yb).item())

    tl = np.mean(train_losses)
    vl = np.mean(val_losses)
    history['train'].append(tl)
    history['val'].append(vl)
    scheduler.step(vl)

    if vl < best_loss:
        best_loss  = vl
        best_epoch = epoch
        torch.save(model.state_dict(), MODELS + 'lstm_best.pt')

    if (epoch + 1) % 3 == 0 or epoch == 0:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1:02d}/{EPOCHS}  train={tl:.4f}  val={vl:.4f}  lr={lr_now:.6f}  (best: ep.{best_epoch+1})')

    if epoch - best_epoch >= PATIENCE:
        print(f'Early stopping na epochi {epoch+1}')
        break

print(f'\nBest epoch: {best_epoch+1},  best val MSE: {best_loss:.4f}')

## 7. Evaluacija na validacijskom setu

In [ ]:
model.load_state_dict(torch.load(MODELS + 'lstm_best.pt', map_location=DEVICE))
model.eval()

preds_scaled = []
with torch.no_grad():
    for Xb, _ in val_loader:
        p = model(Xb.to(DEVICE)).cpu().numpy()
        preds_scaled.extend(p)

preds_scaled = np.array(preds_scaled, dtype=np.float32)

# Inverse transform: samo sales_log kolona
mean_sl  = scaler.mean_[sales_log_idx]
scale_sl = scaler.scale_[sales_log_idx]

lstm_pred_log  = preds_scaled * scale_sl + mean_sl
y_val_log      = y_val        * scale_sl + mean_sl

lstm_pred_sales = np.expm1(lstm_pred_log).clip(min=0)
y_val_sales     = np.expm1(y_val_log)

rmsle = np.sqrt(np.mean((np.log1p(lstm_pred_sales) - np.log1p(y_val_sales))**2))
mae   = np.mean(np.abs(lstm_pred_sales - y_val_sales))

print('LSTM validacijski set (2017-08-01 do 2017-08-15):')
print(f'  RMSLE: {rmsle:.4f}')
print(f'  MAE:   {mae:.2f}')
print(f'  MAE %: {mae / y_val_sales.mean() * 100:.1f}%')

In [ ]:
lgbm_rmsle = 0.3695

print('Poredjenje modela:')
print(f'  Naive baseline (lag_7): RMSLE = 0.5690')
print(f'  LightGBM + Prophet:     RMSLE = {lgbm_rmsle:.4f}')
print(f'  LSTM:                   RMSLE = {rmsle:.4f}')
print()
print('Sledeci korak: ensemble kombinacija u 06_ensemble.ipynb')

## 8. Vizualizacije

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['train'], label='Train MSE (skaliran prostor)', color='steelblue')
ax.plot(history['val'],   label='Val MSE (skaliran prostor)',   color='darkorange')
ax.axvline(best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Best (ep.{best_epoch+1})')
ax.set_title('LSTM - history treniranja')
ax.set_xlabel('Epocha')
ax.set_ylabel('MSE')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
val_results = pd.DataFrame({
    'date':          pd.to_datetime(val_dates_list),
    'store_nbr':     val_stores_list,
    'family':        val_families_list,
    'lstm_pred_log': lstm_pred_log,
    'lstm_pred':     lstm_pred_sales,
    'actual_sales':  y_val_sales,
})

daily_actual = val_results.groupby('date')['actual_sales'].sum()
daily_pred   = val_results.groupby('date')['lstm_pred'].sum()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily_actual.index, daily_actual.values, label='Stvarna prodaja', color='steelblue', linewidth=2)
ax.plot(daily_pred.index,   daily_pred.values,   label='LSTM predikcija', color='darkorange', linewidth=2, linestyle='--')
ax.set_title('Ukupna dnevna prodaja: stvarna vs LSTM (val set)')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Snimanje predikcija za ensemble

In [ ]:
val_results.to_parquet(MODELS + 'lstm_val_preds.parquet', index=False)

with open(MODELS + 'lstm_scaler.pkl', 'wb') as f:
    pickle.dump({
        'scaler':       scaler,
        'seq_features': SEQ_FEATURES,
        'seq_len':      SEQ_LEN,
    }, f)

print('Snimljeno:')
print(f'  models/lstm_val_preds.parquet  ({val_results.shape})')
print(f'  models/lstm_best.pt')
print(f'  models/lstm_scaler.pkl')
print(f'  Val RMSLE: {rmsle:.4f}')